In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql.functions import col, lit,current_date


In [0]:
from utils.api_response import api_job_search
from utils.config import scope_api_name,catalog_name,schema_name,bronze_table
from transformations.ingestion_transformations import create_bronze_df,merge_bronze_df_target

###Get the secrets

In [0]:
scope_name=scope_api_name
api_token_= dbutils.secrets.get(scope=scope_name, key="api_token")
api_url= dbutils.secrets.get(scope=scope_name, key="api_url")
api_host= dbutils.secrets.get(scope=scope_name, key="api_host")

##load Data from API call

- init the Api job search class
- load data from the API

In [0]:
responses=api_job_search(api_token_,api_url,api_host)
raw_data=responses.get_jobs(job_title="data",country="fr",date_posted="all")

##ingest data to the bronze layer

In [0]:
df=create_bronze_df(spark,raw_data)
merged_success,insert,updated,deleted=merge_bronze_df_target(spark,df_source=df,df_target=f"{catalog_name}.{schema_name}.{bronze_table}")

######Create a job parameter to use it later int the pipeline

In [0]:
dbutils.jobs.taskValues.set(key="upsert_rows",value=updated+insert)